In [8]:
import math

class NaiveBayesScratch:
  def __init__(self):
    self.logpriors = {}
    self.word_probs = {}
    self.classes = []
    self.vocab = set()
    
  def fit(self, X, y):
    """
    X: List of strings (sentences).
    y: List of labels (0 or 1).
    """
    
    n_samples = len(X)
    self.classes = set(y)
    
    # 1. Separate data by class
    classes_docs = {c: [] for c in self.classes}
    for doc, label in zip(X, y):
      classes_docs[label].append(doc)
      # Build vocabulary
      for word in doc.split():
        self.vocab.add(word)
        
    vocab_size = len(self.vocab)
    
    # 2. Calculate Statistics
    for c in self.classes:
      docs = classes_docs[c]
      
      # A. Prior: P(Class)
      # We use Logarithms to avoid "Underflow" (multiplying tiny numbers becomes 0)
      # log(a * b) = log(a) + log(b)
      
      self.logpriors[c] = math.log(len(docs) / n_samples)
      
      # B. Likelihood: P(Word|Class)
      total_word_counts = {}
      total_word_in_class = 0
      
      # Count every word in this class's documents
      for doc in docs:
        for word in doc.split():
          total_word_counts[word] = total_word_counts.get(word, 0) + 1
          total_word_in_class += 1
          
      # Calculate Probability for each word in vocab
      self.word_probs[c] = {}
      for word in self.vocab:
        #Get default count 0
        count = total_word_counts.get(word, 0)
        # Laplace Smoothing: (Count + 1) / (Total + VocabSize)
        prob = (count+1) / (total_word_in_class + vocab_size)
        
        self.word_probs[c][word] = math.log(prob)
        
  def predict(self, X):
    predictions = []
    for doc in X:
      class_scores = {}
      
      for c in self.classes:
        # Start with Prior
        score = self.logpriors[c]
        
        # Add Log Probability of each word
        words = doc.split()
        for word in words:
          if word in self.vocab:
            score += self.word_probs[c][word]        
            
        class_scores[c] = score
        
      best_class = max(class_scores, key=class_scores.get)
      predictions.append(best_class)
    return predictions

In [9]:
# --- Usage Example ---
if __name__ == "__main__":
    # 1. Training Data
    X_train = [
        "offer is secret",          # Spam (1)
        "click secret link",        # Spam (1)
        "play sports today",        # Ham (0)
        "sports are healthy"        # Ham (0)
    ]
    y_train = [1, 1, 0, 0]

    model = NaiveBayesScratch()
    model.fit(X_train, y_train)

    # 2. Test
    test_sentence = ["secret sports link"]
    pred = model.predict(test_sentence)
    
    label = "Spam" if pred[0] == 1 else "Ham"
    print(f"Prediction for '{test_sentence[0]}': {label}")

Prediction for 'secret sports link': Spam
